In [7]:
import pandas as pd
from catboost import CatBoostClassifier

import pickle
from geoai.utils_geo.RasterOps import RasterOperations
from geoai.utils_ml.ModelOps import ModelOperations

raster_ops = RasterOperations()
model_ops = ModelOperations()

In [2]:
X_train = pd.read_csv("csv_files/X_train.csv")
X_test = pd.read_csv("csv_files/X_test.csv")
y_train = pd.read_csv("csv_files/y_train.csv")
y_test = pd.read_csv("csv_files/y_test.csv")

# compute indices
X_train = raster_ops.compute_ndvi_using_df(X_train, "NIR", "RED")
X_train = raster_ops.compute_ndbi(X_train, "NIR", "SWIR")
X_train = raster_ops.compute_rei(X_train, "NIR", "BLUE")
X_test = raster_ops.compute_ndvi_using_df(X_test, "NIR", "RED")
X_test = raster_ops.compute_ndbi(X_test, "NIR", "SWIR")
X_test = raster_ops.compute_rei(X_test, "NIR", "BLUE")


# create binary and discrete NDVI
X_train = raster_ops.create_ndvi_bin(X_train, "NDVI")
X_train = raster_ops.create_ndvi_discrete(X_train, "NDVI")
X_test = raster_ops.create_ndvi_bin(X_test, "NDVI")
X_test = raster_ops.create_ndvi_discrete(X_test, "NDVI")


In [5]:
cat = CatBoostClassifier()
# create a pipeline
pipeline = model_ops.make_pipeline(cat)
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('poly',
                                                                   PolynomialFeatures())]),
                                                  ['BLUE', 'GREEN', 'RED',
                                                   'NIR', 'SWIR', 'NDVI',
                                                   'NDBI', 'REI']),
                                                 ('onehot',
                                                  OneHotEncoder(dtype=<class 'int'>),
                                                  ['NDVI_bin']),
                                                 ('ordinal',
                                                  OrdinalEncoder(categories=[['low_veg',
                                                                              'medium_veg',
                                                                              'high_veg']],
                                                                 dtype=<class 'int'>),
                                                  ['NDVI_dis'])])),
                ('scale', MinMaxScaler()),
                ('dim_reduce', LinearDiscriminantAnalysis(n_components=3)),
                ('classifier',
                 <catboost.core.CatBoostClassifier object at 0x00000223E5C6E410>)])

In [6]:
pipeline.fit(X_train, y_train)

# Predict the labels of the test set
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

# Calculate the accuracy of the VotingClassifier
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_train, y_train_pred)}")
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_test, y_test_pred)}")

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Learning rate set to 0.081836
0:	learn: 1.1969704	total: 119ms	remaining: 1m 58s
1:	learn: 1.0590574	total: 125ms	remaining: 1m 2s
2:	learn: 0.9459788	total: 130ms	remaining: 43.4s
3:	learn: 0.8541431	total: 136ms	remaining: 33.8s
4:	learn: 0.7805628	total: 141ms	remaining: 28.1s
5:	learn: 0.7154467	total: 147ms	remaining: 24.4s
6:	learn: 0.6583576	total: 153ms	remaining: 21.7s
7:	learn: 0.6100062	total: 158ms	remaining: 19.6s
8:	learn: 0.5677890	total: 165ms	remaining: 18.2s
9:	learn: 0.5302573	total: 171ms	remaining: 16.9s
10:	learn: 0.4964002	total: 177ms	remaining: 15.9s
11:	learn: 0.4661938	total: 182ms	remaining: 15s
12:	learn: 0.4387149	total: 188ms	remaining: 14.3s
13:	learn: 0.4156014	total: 194ms	remaining: 13.7s
14:	learn: 0.3939309	total: 201ms	remaining: 13.2s
15:	learn: 0.3748572	total: 207ms	remaining: 12.7s
16:	learn: 0.3585668	total: 221ms	remaining: 12.8s
17:	learn: 0.3426198	total: 227ms	remaining: 12.4s
18:	learn: 0.3281979	total: 233ms	remaining: 12s
19:	learn: 0.3

In [8]:
X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])

# train the model using the best hyperparameters and the whole dataset
pipeline.fit(X_all, y_all)
with open('cat.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Learning rate set to 0.082773
0:	learn: 1.1929422	total: 8.5ms	remaining: 8.49s
1:	learn: 1.0508653	total: 15.3ms	remaining: 7.63s
2:	learn: 0.9380031	total: 22.9ms	remaining: 7.62s
3:	learn: 0.8453197	total: 88.7ms	remaining: 22.1s
4:	learn: 0.7716176	total: 102ms	remaining: 20.4s
5:	learn: 0.7059052	total: 139ms	remaining: 23.1s
6:	learn: 0.6494486	total: 199ms	remaining: 28.2s
7:	learn: 0.6006390	total: 212ms	remaining: 26.3s
8:	learn: 0.5580940	total: 229ms	remaining: 25.2s
9:	learn: 0.5202491	total: 245ms	remaining: 24.2s
10:	learn: 0.4863446	total: 276ms	remaining: 24.8s
11:	learn: 0.4570740	total: 292ms	remaining: 24s
12:	learn: 0.4298528	total: 308ms	remaining: 23.4s
13:	learn: 0.4061087	total: 346ms	remaining: 24.4s
14:	learn: 0.3851946	total: 359ms	remaining: 23.6s
15:	learn: 0.3664170	total: 365ms	remaining: 22.5s
16:	learn: 0.3489190	total: 371ms	remaining: 21.5s
17:	learn: 0.3331488	total: 379ms	remaining: 20.7s
18:	learn: 0.3197099	total: 391ms	remaining: 20.2s
19:	learn: